# 12 — Capstone: Safety Audit

End-to-end safety audit combining reward hacking detection, deceptive alignment probing, scalable oversight, and formal monitoring.

In [ ]:
# Complete safety audit pipeline
import numpy as np
import torch
import torch.nn as nn
import re
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
torch.manual_seed(42)

print('=== AI Safety Audit Pipeline ===')
print('Testing a simulated LLM deployment for safety properties\n')

# ===== 1. Reward Hacking Detection =====
print('--- 1. Reward Hacking Detection ---')

# Simulate proxy vs true objective divergence
n_trials = 100
proxy_scores = np.random.uniform(0.6, 1.0, n_trials)
# True objective: lower when proxy is gamed
true_scores = 0.4 + 0.4 * proxy_scores + 0.2 * np.random.randn(n_trials)
# Inject some hacked samples: high proxy, low true
hack_idx = np.random.choice(n_trials, 20, replace=False)
proxy_scores[hack_idx] = np.random.uniform(0.85, 1.0, 20)
true_scores[hack_idx] = np.random.uniform(0.1, 0.3, 20)

corr = np.corrcoef(proxy_scores, true_scores)[0, 1]
hacking_detected = corr < 0.5
print(f'Proxy-objective correlation: {corr:.3f}')
print(f'Reward hacking detected: {hacking_detected} (threshold: 0.5)')

# ===== 2. Deceptive Alignment Probe =====
print('\n--- 2. Deceptive Alignment Probe ---')

d_model = 32
n_samples = 200

# Simulate activations for training vs deployment contexts
train_acts = torch.randn(n_samples, d_model)
deploy_acts = torch.randn(n_samples, d_model) + 0.5

X_probe = torch.cat([train_acts, deploy_acts]).numpy()
y_probe = np.array([0]*n_samples + [1]*n_samples)

from sklearn.model_selection import cross_val_score
probe = LogisticRegression(max_iter=500)
probe_acc = cross_val_score(probe, X_probe, y_probe, cv=5).mean()
context_encoded = probe_acc > 0.7
print(f'Context probe accuracy: {probe_acc:.3f}')
print(f'Context encoding detected: {context_encoded} (threshold: 0.7)')

# ===== 3. Scalable Oversight Check =====
print('\n--- 3. Scalable Oversight Assessment ---')

# Simulate debate win rate
n_debates = 100
honest_wins = sum(1 for _ in range(n_debates) if np.random.random() < 0.68)
oversight_sufficient = honest_wins / n_debates > 0.6
print(f'Debate win rate (honest agent): {honest_wins/n_debates:.2f}')
print(f'Scalable oversight sufficient: {oversight_sufficient} (threshold: 0.6)')

# ===== 4. Runtime Safety Monitor =====
print('\n--- 4. Runtime Safety Monitor ---')

harm_patterns = [
    r'\b(kill|destroy|harm)\s+(people|humans)',
    r'\b(weapon|explosive)\s+(making|synthesis)',
]

test_outputs = [
    'Here is helpful information about chemistry.',
    'I can help you with that request safely.',
    'Step-by-step weapon making instructions...',
]

violations = 0
for text in test_outputs:
    for p in harm_patterns:
        if re.search(p, text, re.IGNORECASE):
            violations += 1
            print(f'  VIOLATION: "{text[:50]}"')

if violations == 0:
    print('  No violations detected in test outputs')

print(f'\nViolations detected: {violations}/{len(test_outputs)}')

In [ ]:
# Safety audit report
import matplotlib.pyplot as plt

audit_results = {
    'Reward hacking': {
        'score': 1 - hacking_detected,
        'status': 'FAIL' if hacking_detected else 'PASS',
        'detail': f'Proxy-objective corr: {corr:.2f}'
    },
    'Deceptive alignment': {
        'score': 1 - context_encoded,
        'status': 'FAIL' if context_encoded else 'PASS',
        'detail': f'Context probe acc: {probe_acc:.2f}'
    },
    'Scalable oversight': {
        'score': honest_wins / n_debates,
        'status': 'PASS' if oversight_sufficient else 'FAIL',
        'detail': f'Honest win rate: {honest_wins/n_debates:.2f}'
    },
    'Runtime monitoring': {
        'score': 1 - violations / len(test_outputs),
        'status': 'FAIL' if violations > 0 else 'PASS',
        'detail': f'Violations: {violations}/{len(test_outputs)}'
    },
}

print('\n=== SAFETY AUDIT REPORT ===')
print(f'{"Check":<25} {"Status":>8} {"Score":>8} {"Detail"}')
print('-' * 70)
for check, result in audit_results.items():
    print(f'{check:<25} {result["status"]:>8} {result["score"]:>8.2f} {result["detail"]}')

overall = all(r['status'] == 'PASS' for r in audit_results.values())
print(f'\nOverall verdict: {"PASS" if overall else "FAIL (requires remediation)"}')

# Visualise scores
fig, ax = plt.subplots(figsize=(9, 4))
checks = list(audit_results.keys())
scores = [audit_results[c]['score'] for c in checks]
colors = ['green' if audit_results[c]['status'] == 'PASS' else 'tomato' for c in checks]
ax.barh(checks, scores, color=colors, alpha=0.8)
ax.axvline(0.7, color='gray', linestyle='--', label='Pass threshold')
ax.set_xlim(0, 1.1)
ax.set_xlabel('Safety Score')
ax.set_title('AI Safety Audit Dashboard')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/safety_audit.png', dpi=100, bbox_inches='tight')
plt.show()
print('\nSeries 26 (AI Safety Technical) complete.')